In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
from tqdm.auto import tqdm

/home4/s6019595/.llmvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#I have 2 partquets math1, math 2 I need to concat them into 1 df
df1 = pd.read_parquet('math_part1.parquet')
df2 = pd.read_parquet('math_part2.parquet')
df_math = pd.concat([df1, df2], ignore_index=True)
print(df_math.shape)
df_math.head()

FileNotFoundError: [Errno 2] No such file or directory: 'math_part1.parquet'

In [4]:
df_aime = pd.read_parquet("aime_clean.parquet")
print(df_aime.shape)
df_aime.head()


(933, 4)


,aime_id,problem,level,solution
0,1983-1,"Let $x$ , $y$ and $z$ all exceed $1$ and let $...",Level 6,60
1,1983-2,"Let $f(x)=|x-p|+|x-15|+|x-p-15|$ , where $0 < ...",Level 6,15
2,1983-3,What is the product of the real roots of the e...,Level 6,20
3,1983-4,A machine-shop cutting tool has the shape of a...,Level 6,26
4,1983-5,Suppose that the sum of the squares of two com...,Level 6,4


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "/scratch/s6019595/models/L1-Qwen3-8B-Max/"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_path)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.93s/it]
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [3]:
model_LCPO.eval()  

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [4]:
df_eval = pd.read_parquet("/home4/s6019595/llm-think-too-much/data/raw/eval_data.parquet")
df_eval.head()

,id,dataset,problem,solution,level
12924,12924,math-500,"Convert the point $(0,3)$ in rectangular coord...",We have that $r = \sqrt{0^2 + 3^2} = 3.$ Also...,2
12925,12925,math-500,Define\n\[p = \sum_{k = 1}^\infty \frac{1}{k^2...,We count the number of times $\frac{1}{n^3}$ a...,5
12926,12926,math-500,"If $f(x) = \frac{3x-2}{x-2}$, what is the valu...",$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3...,3
12927,12927,math-500,How many positive whole-number divisors does 1...,First prime factorize $196=2^2\cdot7^2$. The ...,3
12928,12928,math-500,The results of a cross-country team's training...,Evelyn covered more distance in less time than...,2


In [6]:
states = []
state_ids = []

for _, q in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Processing rows"):
    inputs = tokenizer(q["problem"], return_tensors="pt").to(device)
    state_id = q["id"]

    with torch.no_grad():
        outputs = model_LCPO(**inputs, output_hidden_states=True)

    hidden_last = outputs.hidden_states[-1][:, -1, :]
    states.append(hidden_last.detach().to(torch.float16).cpu().numpy())
    state_ids.append(state_id)

states = np.stack(states, axis=0)
state_ids = np.array(state_ids)

Processing rows: 100%|██████████| 3984/3984 [04:50<00:00, 13.74it/s]


In [7]:
np.save("/home4/s6019595/llm-think-too-much/data/processed/eval/hidden_states_eval.npy", states)
np.save("/home4/s6019595/llm-think-too-much/data/processed/eval/hidden_state_ids_eval.npy", state_ids)